In [1]:
from rdflib import Graph, RDF
from rdflib.namespace import OWL, XSD, SKOS, RDFS
import pandas as pd

In [2]:
graph = Graph()
graph.parse('../data/input/AI-RHEUM.ttl', format='ttl')

<Graph identifier=N24cdac61abcc4d029daf3d637d82d063 (<class 'rdflib.graph.Graph'>)>

In [3]:
print(f"Graph g has {len(graph)} statements.")

Graph g has 5968 statements.


In [4]:
import itertools

class Indexer(object):
    def __init__(self,it):
        self.it = it
    
    def __iter__(self):
        return self.it
    
    def __getitem__(self,index):
        try:
            return next(itertools.islice(self.it,index,index+1))
        except TypeError:
            return list(itertools.islice(self.it,index.start,index.stop,index.step))
        

In [19]:
dic ={}

for owlClass in graph.subjects(RDF.type, OWL.Class):
    for notation in  graph.objects(owlClass, SKOS.notation):
        dic[str(owlClass)] = {}
    for label in graph.objects(owlClass, SKOS.prefLabel):
        dic[str(owlClass)]["label"] = str(label)
    dic[str(owlClass)]["synonyms"] = []
    dic[str(owlClass)]["semantic_types"] = []
    for synonyms in graph.objects(owlClass, SKOS.altLabel):
        dic[str(owlClass)]["synonyms"].append(str(synonyms))
    for semantic_types in graph.objects(owlClass, RDFS.subClassOf):
        dic[str(owlClass)]["semantic_types"].append(str(semantic_types))


dic["http://purl.bioontology.org/ontology/AIR/BMNST"]

{'label': 'Lumbar spine pain and/or morning stiffness or night pain of the back, improving with motion',
 'synonyms': [],
 'semantic_types': ['http://purl.bioontology.org/ontology/AIR/MFSPI',
  'http://purl.bioontology.org/ontology/AIR/U000083']}

In [12]:
dictData = pd.DataFrame(columns=['ID', 'Label'])
for owlClass in graph.subjects(RDF.type, OWL.Class):
    for prefLabel in graph.objects(owlClass, SKOS.prefLabel):
        dictData.loc[len(dictData)] = [owlClass, prefLabel]
    for altLabel in graph.objects(owlClass, SKOS.altLabel):
        dictData.loc[len(dictData)] = [owlClass, altLabel]
        
dictData

,ID,Label
0,http://purl.bioontology.org/ontology/AIR/LBPN,Lumbar spine pain > 3 months
1,http://purl.bioontology.org/ontology/AIR/AGE,Patient age at this workup
2,http://purl.bioontology.org/ontology/AIR/BMNST,Lumbar spine pain and/or morning stiffness or ...
3,http://purl.bioontology.org/ontology/AIR/CPD,Pseudo-gout
4,http://purl.bioontology.org/ontology/AIR/CPD,calcium pyrophosphate deposition disease
...,...,...
807,http://purl.bioontology.org/ontology/STY/T085,Molecular Sequence
808,http://purl.bioontology.org/ontology/STY/T010,Vertebrate
809,http://purl.bioontology.org/ontology/STY/T016,Human
810,http://purl.bioontology.org/ontology/STY/T021,Fully Formed Anatomical Structure


In [6]:
dictData[dictData['ID'].astype(str)=='http://purl.bioontology.org/ontology/AIR/CPD']

,ID,Label


In [156]:
dictData = pd.DataFrame(columns=['ID', 'Label'])
for elem in graph.subject_objects(SKOS.prefLabel):
    dictData.loc[len(dictData)] = [elem[0], elem[1]]
for elem in graph.subject_objects(SKOS.altLabel):
    dictData.loc[len(dictData)] = [elem[0], elem[1]]

In [8]:
dictData[dictData['ID'].astype(str)=='http://purl.bioontology.org/ontology/AIR/CPD']

,ID,Label


In [9]:
dictData = pd.DataFrame(columns=['ID', 'Label'])

labelsQuery = """
SELECT DISTINCT ?id ?label
WHERE {
  { ?id a owl:Class .
  ?id skos:prefLabel ?label }
  UNION
  { ?id a owl:Class .
  ?id skos:altLabel ?label }
}
"""

qres = graph.query(labelsQuery)
for row in qres:
    dictData.loc[len(dictData)] = [row.id, row.label]

dictData

,ID,Label
0,http://purl.bioontology.org/ontology/AIR/LBPN,Lumbar spine pain > 3 months
1,http://purl.bioontology.org/ontology/AIR/AGE,Patient age at this workup
2,http://purl.bioontology.org/ontology/AIR/BMNST,Lumbar spine pain and/or morning stiffness or ...
3,http://purl.bioontology.org/ontology/AIR/CPD,Pseudo-gout
4,http://purl.bioontology.org/ontology/AIR/DXNA,None of the above
...,...,...
807,http://purl.bioontology.org/ontology/STY/T203,Drug Delivery Device
808,http://purl.bioontology.org/ontology/AIR/CPD,calcium pyrophosphate deposition disease
809,http://purl.bioontology.org/ontology/AIR/DJD,primary OA
810,http://purl.bioontology.org/ontology/AIR/JR3,Still's disease


In [10]:
dictData[dictData['ID'].astype(str)=='http://purl.bioontology.org/ontology/AIR/CPD']

,ID,Label
